# 🟣 DumpGuard Detection - Credential Guard Bypass

![Weekly Purple Team](https://img.shields.io/badge/Weekly%20Purple%20Team-Threat%20Hunting-purple?style=for-the-badge)
![MITRE ATT&CK](https://img.shields.io/badge/MITRE%20ATT%26CK-T1003.001-red?style=for-the-badge)

**YouTube**: [Weekly Purple Team](https://youtube.com/@WeeklyPurpleTeam)

---

## 📋 Overview

This notebook provides detection strategies for **DumpGuard**, a credential dumping tool released by SpecterOps in October 2025. DumpGuard can extract NTLMv1 hashes from modern Windows systems, **even when Credential Guard is enabled**.

### What Makes DumpGuard Different?

Traditional credential dumping tools like Mimikatz directly access LSASS memory, which is blocked by Credential Guard. DumpGuard bypasses this by abusing the **Remote Credential Guard protocol** - a legitimate Windows feature designed to protect credentials during remote connections.

| Feature | Traditional LSASS Dump | DumpGuard |
|---------|----------------------|------------|
| Blocked by Credential Guard | ✅ Yes | ❌ No |
| Requires LSASS Memory Access | ✅ Yes | ❌ No |
| Output Format | NTLM Hash / Plaintext | NTLMv1 Response |
| Privilege Required | Admin/SYSTEM | None (self) / SYSTEM (all) |

## 🎯 MITRE ATT&CK Mapping

| Tactic | Technique | Sub-Technique | Description |
|--------|-----------|---------------|-------------|
| Credential Access | [T1003](https://attack.mitre.org/techniques/T1003/) | [.001 - LSASS Memory](https://attack.mitre.org/techniques/T1003/001/) | OS Credential Dumping |
| Lateral Movement | [T1550](https://attack.mitre.org/techniques/T1550/) | [.002 - Pass the Hash](https://attack.mitre.org/techniques/T1550/002/) | NTLMv1 hash relay |
| Defense Evasion | [T1562](https://attack.mitre.org/techniques/T1562/) | [.001 - Disable or Modify Tools](https://attack.mitre.org/techniques/T1562/001/) | Bypasses Credential Guard |

## 🔴 Attack Overview

### DumpGuard Modes of Operation

DumpGuard supports three attack modes:

#### 1. Self Mode (Unprivileged)
```
DumpGuard.exe /mode:self /domain:CORP /username:svc_spn /password:Password123
```
- Dumps NTLMv1 hash for **current user only**
- Requires credentials for an SPN-enabled account
- Works even with Credential Guard enabled
- **No special privileges required**

#### 2. All Mode with Remote Credential Guard (SYSTEM)
```
DumpGuard.exe /mode:all /domain:CORP /username:svc_spn /password:Password123
```
- Dumps NTLMv1 hashes for **all authenticated users**
- Impersonates tokens from running processes
- Works even with Credential Guard enabled
- **Requires SYSTEM privileges**

#### 3. All Mode with MSV1_0 (SYSTEM)
```
DumpGuard.exe /mode:all
```
- Uses Microsoft v1 authentication package
- Only works when Credential Guard is **disabled**
- **Requires SYSTEM privileges**

## 🔵 Detection Strategy

### Required Data Sources

| Data Source | Event ID | Purpose |
|-------------|----------|----------|
| Windows Security | 4688 | Process Creation |
| Windows Security | 4656 | Handle to Object Requested |
| Windows Security | 4663 | Object Access |
| Sysmon | 1 | Process Creation |
| Sysmon | 10 | Process Access |
| Sysmon | 17/18 | Pipe Created/Connected |

### Detection Approaches

1. **Command Line Detection** - Look for DumpGuard execution patterns
2. **LSASS Handle Access** - Monitor suspicious access to LSASS from user directories
3. **Named Pipe Activity** - Remote Credential Guard uses specific named pipes
4. **Token Impersonation** - Detect mass token impersonation activity

---
## 🔍 Detection #1: DumpGuard Command Line Arguments

Detect DumpGuard execution by identifying its characteristic command line arguments including `/mode:self` and `/mode:all`.

### Elastic / OpenSearch (Lucene)
```lucene
event.code:(4688 OR 1) AND process.command_line:(*\/mode\:self* OR *\/mode\:all*)
```

### Microsoft Sentinel (KQL)
```kql
// DumpGuard Command Line Detection
let DumpGuardPatterns = dynamic(["/mode:self", "/mode:all"]);
union SecurityEvent, SysmonEvent
| where EventID in (4688, 1)
| where CommandLine has_any (DumpGuardPatterns)
| project 
    TimeGenerated,
    Computer,
    Account,
    ProcessName = iff(EventID == 4688, NewProcessName, Image),
    CommandLine,
    ParentProcessName = iff(EventID == 4688, ParentProcessName, ParentImage),
    EventID
| extend 
    DumpGuardMode = case(
        CommandLine contains "/mode:all", "All Sessions",
        CommandLine contains "/mode:self", "Self Only",
        "Unknown"
    )
```

### Splunk (SPL)
```spl
index=windows (EventCode=4688 OR EventCode=1)
| search CommandLine="*/mode:self*" OR CommandLine="*/mode:all*"
| eval DumpGuardMode=case(
    like(CommandLine, "%/mode:all%"), "All Sessions",
    like(CommandLine, "%/mode:self%"), "Self Only",
    true(), "Unknown"
)
| table _time, host, user, Image, CommandLine, ParentImage, DumpGuardMode
```

### CrowdStrike Falcon (NG-SIEM)
```
#event_simpleName=ProcessRollup2
| CommandLine=/\/mode:(self|all)/i
| select([timestamp, ComputerName, UserName, ImageFileName, CommandLine, ParentBaseFileName])
```

### Cortex XSIAM (XQL)
```xql
dataset = xdr_data
| filter event_type = PROCESS and event_sub_type = PROCESS_START
| filter action_process_command_line contains "/mode:self" or action_process_command_line contains "/mode:all"
| fields _time, agent_hostname, actor_effective_username, action_process_image_path, action_process_command_line, actor_process_image_path
```

### Sigma Rule
```yaml
title: DumpGuard Credential Dumping Tool Execution
id: a8d7e5c2-3f4b-4a1d-9e6c-7b8f2d1e3c4a
status: experimental
description: Detects execution of DumpGuard, a tool that extracts NTLMv1 hashes even when Credential Guard is enabled
references:
    - https://github.com/bytewreck/DumpGuard
    - https://specterops.io/blog/2025/10/23/catching-credential-guard-off-guard/
author: Weekly Purple Team
date: 2025/02/06
tags:
    - attack.credential_access
    - attack.t1003.001
    - attack.t1550.002
logsource:
    category: process_creation
    product: windows
detection:
    selection_mode:
        CommandLine|contains:
            - '/mode:self'
            - '/mode:all'
    selection_params:
        CommandLine|contains:
            - '/domain:'
            - '/username:'
            - '/password:'
            - '/spn:'
    condition: selection_mode or (selection_mode and selection_params)
falsepositives:
    - Legitimate security testing with proper authorization
level: high
```

---
## 🔍 Detection #2: LSASS Handle Access from User Directories

Detect when processes running from user-writable directories attempt to access LSASS. This is a strong indicator of credential theft tools.

### Elastic / OpenSearch (Lucene)
```lucene
event.code:4656 AND winlog.event_data.ObjectName:*lsass* AND winlog.event_data.ProcessName:*C\:\\Users*
```

### Microsoft Sentinel (KQL)
```kql
// LSASS Handle Access from User Directories
SecurityEvent
| where EventID == 4656
| where ObjectName contains "lsass"
| where ProcessName startswith "C:\\Users"
| project 
    TimeGenerated,
    Computer,
    Account,
    ProcessName,
    ObjectName,
    AccessMask,
    AccessList
| extend 
    UserPath = extract(@"C:\\Users\\([^\\]+)", 1, ProcessName),
    SuspiciousAccess = iff(AccessMask in ("0x1010", "0x1410", "0x1F1FFF"), true, false)
```

### Splunk (SPL)
```spl
index=windows EventCode=4656 ObjectName="*lsass*" ProcessName="C:\\Users\\*"
| eval AccessType=case(
    AccessMask="0x1010", "Read",
    AccessMask="0x1410", "Read+Write",
    AccessMask="0x1F1FFF", "Full Access",
    true(), AccessMask
)
| rex field=ProcessName "C:\\\\Users\\\\(?<UserProfile>[^\\\\]+)"
| table _time, host, user, ProcessName, UserProfile, ObjectName, AccessType
```

### CrowdStrike Falcon (NG-SIEM)
```
#event_simpleName=ProcessRollup2 OR #event_simpleName=LsassHandleAccess
| ImageFileName=/C:\\Users\\/i
| TargetProcessName=/lsass/i
| select([timestamp, ComputerName, UserName, ImageFileName, TargetProcessName, DesiredAccess])
```

### Cortex XSIAM (XQL)
```xql
dataset = xdr_data
| filter event_type = KERNEL and action_type = "READ_PROCESS_MEMORY"
| filter actor_process_image_path contains "\\Users\\"
| filter target_process_image_path contains "lsass"
| fields _time, agent_hostname, actor_effective_username, actor_process_image_path, target_process_image_path
```

### Sigma Rule
```yaml
title: LSASS Handle Access from User Directory
id: b9e8f7a6-5d4c-3b2a-1e0f-9c8d7b6a5e4d
status: experimental
description: Detects processes running from user directories attempting to access LSASS, indicating potential credential theft
references:
    - https://github.com/bytewreck/DumpGuard
author: Weekly Purple Team
date: 2025/02/06
tags:
    - attack.credential_access
    - attack.t1003.001
logsource:
    product: windows
    service: security
detection:
    selection:
        EventID: 4656
        ObjectName|contains: 'lsass'
        ProcessName|startswith: 'C:\\Users\\'
    condition: selection
falsepositives:
    - Legitimate security software running from user directories
    - Security assessments
level: high
```

---
## 🔍 Detection #3: Remote Credential Guard Named Pipe Activity

DumpGuard abuses the Remote Credential Guard protocol which uses specific named pipes for communication. Monitor for suspicious access to these pipes.

### Microsoft Sentinel (KQL)
```kql
// Remote Credential Guard Named Pipe Access
// Sysmon Event ID 17 (Pipe Created) and 18 (Pipe Connected)
Event
| where Source == "Microsoft-Windows-Sysmon"
| where EventID in (17, 18)
| extend EventData = parse_xml(EventData)
| extend PipeName = tostring(EventData.DataItem.EventData.Data[4])
| where PipeName has_any ("NtlmCredIso", "CredentialGuard", "LsaIso")
| project 
    TimeGenerated,
    Computer,
    EventID,
    PipeName,
    ProcessName = tostring(EventData.DataItem.EventData.Data[5]),
    ProcessId = tostring(EventData.DataItem.EventData.Data[6])
```

### Splunk (SPL)
```spl
index=windows sourcetype="XmlWinEventLog:Microsoft-Windows-Sysmon/Operational" 
    (EventCode=17 OR EventCode=18)
| spath output=PipeName path=Event.EventData.Data{@Name="PipeName"}
| search PipeName="*NtlmCredIso*" OR PipeName="*CredentialGuard*" OR PipeName="*LsaIso*"
| table _time, host, user, PipeName, Image, ProcessId
```

### Elastic / OpenSearch (Lucene)
```lucene
event.code:(17 OR 18) AND winlog.event_data.PipeName:(*NtlmCredIso* OR *CredentialGuard* OR *LsaIso*)
```

---
## 🔍 Detection #4: DumpGuard Binary Indicators

Detect DumpGuard based on binary name patterns and associated file hashes (when available).

### Microsoft Sentinel (KQL)
```kql
// DumpGuard Binary Detection
let DumpGuardNames = dynamic(["DumpGuard.exe", "dumpguard"]);
union SecurityEvent, SysmonEvent
| where EventID in (4688, 1)
| where ProcessName has_any (DumpGuardNames) 
    or CommandLine has_any (DumpGuardNames)
    or OriginalFileName has_any (DumpGuardNames)
| project 
    TimeGenerated,
    Computer,
    Account,
    ProcessName,
    CommandLine,
    ParentProcessName,
    Hashes
```

### Splunk (SPL)
```spl
index=windows (EventCode=4688 OR EventCode=1)
| search Image="*DumpGuard*" OR CommandLine="*DumpGuard*" OR OriginalFileName="*DumpGuard*"
| table _time, host, user, Image, CommandLine, ParentImage, Hashes
```

---
## 🔍 Detection #5: Token Impersonation Burst (All Mode Detection)

When DumpGuard runs in `/mode:all`, it rapidly impersonates tokens from multiple processes. Detect this burst of impersonation activity.

### Microsoft Sentinel (KQL)
```kql
// Mass Token Impersonation Detection
// Windows Security Event 4648 - Logon with explicit credentials
SecurityEvent
| where EventID == 4648
| summarize 
    ImpersonationCount = count(),
    UniqueTargetUsers = dcount(TargetUserName),
    TargetUsers = make_set(TargetUserName)
    by bin(TimeGenerated, 1m), Computer, SubjectUserName, ProcessName
| where ImpersonationCount > 10 and UniqueTargetUsers > 3
| project 
    TimeGenerated,
    Computer,
    SubjectUserName,
    ProcessName,
    ImpersonationCount,
    UniqueTargetUsers,
    TargetUsers
```

### Splunk (SPL)
```spl
index=windows EventCode=4648
| bin _time span=1m
| stats 
    count as ImpersonationCount,
    dc(TargetUserName) as UniqueTargetUsers,
    values(TargetUserName) as TargetUsers
    by _time, host, SubjectUserName, ProcessName
| where ImpersonationCount > 10 AND UniqueTargetUsers > 3
| table _time, host, SubjectUserName, ProcessName, ImpersonationCount, UniqueTargetUsers, TargetUsers
```

---
## 📊 Hunting Queries

### Hunt #1: Processes with Credential-Related Arguments (Elastic)

Broad hunt for processes with credential-dumping related command line arguments.

```lucene
event.code:(4688 OR 1) 
AND process.command_line:(
    *\/domain\:* OR 
    *\/username\:* OR 
    *\/password\:* OR
    *\/spn\:* OR
    *sekurlsa* OR
    *credential*
)
AND NOT process.executable:(*Microsoft* OR *Windows\\System32*)
```

### Hunt #2: Unusual LSASS Access Patterns (KQL)

```kql
// Hunt for unusual patterns of LSASS access
SecurityEvent
| where EventID in (4656, 4663)
| where ObjectName contains "lsass"
| where ProcessName !endswith "\\MsMpEng.exe"  // Exclude Defender
| where ProcessName !endswith "\\csrss.exe"  
| where ProcessName !endswith "\\svchost.exe"
| where ProcessName !endswith "\\lsass.exe"
| summarize 
    AccessCount = count(),
    AccessTimes = make_list(TimeGenerated)
    by Computer, ProcessName, Account
| where AccessCount > 1
| sort by AccessCount desc
```

### Hunt #3: SPN Account Usage Anomalies (KQL)

DumpGuard requires an SPN-enabled account. Hunt for unusual authentication patterns with service accounts.

```kql
// Unusual SPN account authentication patterns
SecurityEvent
| where EventID == 4624  // Successful logon
| where TargetUserName matches regex @"^svc_|^service_|_svc$"  // Service account patterns
| where LogonType in (2, 9, 10, 11)  // Interactive, NewCredentials, RemoteInteractive, CachedInteractive
| summarize 
    LogonCount = count(),
    SourceIPs = make_set(IpAddress),
    SourceHosts = make_set(WorkstationName)
    by TargetUserName, Computer, LogonType
| where LogonCount > 5
```

---
## 🛡️ Mitigation Recommendations

### 1. Enable Windows Defender Credential Guard
While DumpGuard can bypass Credential Guard for NTLMv1, it still provides significant protection against traditional attacks.

```powershell
# Check Credential Guard status
Get-CimInstance -ClassName Win32_DeviceGuard -Namespace root\Microsoft\Windows\DeviceGuard
```

### 2. Restrict SPN Account Permissions
- Limit the number of accounts with SPNs
- Use Group Managed Service Accounts (gMSA) where possible
- Regularly audit SPN assignments

```powershell
# Find all accounts with SPNs
Get-ADUser -Filter {ServicePrincipalNames -like "*"} -Properties ServicePrincipalNames
```

### 3. Enable LSA Protection (RunAsPPL)
```
HKLM\SYSTEM\CurrentControlSet\Control\Lsa
RunAsPPL = 1 (DWORD)
```

### 4. Monitor for NTLMv1 Usage
- Disable NTLMv1 where possible
- Monitor Event ID 4624 for NTLMv1 authentication

### 5. Implement Application Control
- Block execution of unauthorized binaries
- Use Windows Defender Application Control (WDAC)
- Restrict execution from user-writable directories

---
## 📚 References

- [DumpGuard GitHub Repository](https://github.com/bytewreck/DumpGuard)
- [SpecterOps Blog: Catching Credential Guard Off Guard](https://specterops.io/blog/2025/10/23/catching-credential-guard-off-guard/)
- [MITRE ATT&CK T1003.001 - LSASS Memory](https://attack.mitre.org/techniques/T1003/001/)
- [Oliver Lyak - Pass the Challenge: Defeating Windows Defender Credential Guard](https://research.ifcr.dk/pass-the-challenge-defeating-windows-defender-credential-guard-31a892eee22)
- [Microsoft - Detecting and Preventing LSASS Credential Dumping Attacks](https://www.microsoft.com/en-us/security/blog/2022/10/05/detecting-and-preventing-lsass-credential-dumping-attacks/)

---
## ⚠️ Disclaimer

This notebook is provided for **educational and authorized security testing purposes only**. The detection strategies and queries should be used to improve your organization's security posture. Always ensure you have proper authorization before conducting security assessments.

---

**Weekly Purple Team** | [YouTube](https://youtube.com/@WeeklyPurpleTeam) | *Learn to attack. Learn to defend. Stay purple.* 🟣